# 2. Coordinates: CRS, projections, units, heights: exercises

Exercises of [chapter 2](../02_coordinates.md). Run the setup cell first.

Work in a copy of this notebook: it is regenerated by `docs/manual/tools/build_notebooks.py`.

In [ ]:
# Setup: find the repository root, make the sample data root if needed, use the sample's site registry
import os, subprocess, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))
SAMPLE = ROOT / "output/manual/sample_root"
if not SAMPLE.exists():
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"
EXTRACT = ROOT / "docs/manual/data/montana_extract"
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")
import warnings; warnings.simplefilter("ignore")
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import numpy as np
import pyogrio
import rasterio
from pyproj import CRS, Transformer
from rasterio.warp import transform

**1. Read a CRS.** Open `ortho_2023-07-12.tif` of the sample dataset with rasterio. Print `crs`, `crs.to_epsg()` and `crs.to_wkt()`. Is it geographic or projected? What are the units of `res`?

In [ ]:
with rasterio.open(DATASET / "ortho_2023-07-12.tif") as r:
    ...  # your code here

**2. Degrees and metres.** Transform the top corners of the sample orthomosaic to EPSG:4326 with `pyproj.Transformer`. How wide is the image in degrees of longitude, and in metres? Explain the ratio.

In [ ]:
with rasterio.open(DATASET / "ortho_2023-07-12.tif") as r:
    b = r.bounds
to_lonlat = Transformer.from_crs("EPSG:32612", "EPSG:4326", always_xy=True)
...  # your code here

**3. No CRS.** Open `dem_2022.asc`. What does rasterio report as its CRS? Where in the sample's `metadata.xml` would you find the CRS to put in the summary?

In [ ]:
with rasterio.open(DATASET / "dem_2022.asc") as r:
    ...  # your code here

**4. Heights in feet.** Open `2018-07-26/annotations/dams.gpkg` of the Montana extract with `pyogrio.read_info`. Split its CRS into horizontal and vertical parts. What is the vertical datum, and its unit? How many metres is 5 US survey feet?

In [ ]:
info = pyogrio.read_info(EXTRACT / "2018-07-26/annotations/dams.gpkg")
...  # your code here

**5. Two PROJ versions.** Take the first vertex of a dam in the extract (the dams are MultiPolygons: `geometry.iloc[0].geoms[0].exterior.coords[0]`). Transform it to EPSG:4326 with `pyproj` and with `rasterio.warp.transform`, first using the CRS written in the file, then using `+proj=eqc +lat_ts=-52.28859923 +datum=WGS84 +units=m`. Which pairs agree and which are 25 km apart? Why does the file in the extract not show the problem?

In [ ]:
path = EXTRACT / "2018-07-26/annotations/dams.gpkg"
dams = pyogrio.read_dataframe(path)
x, y = dams.geometry.iloc[0].geoms[0].exterior.coords[0][:2]
...  # your code here

**6. Find it in the code.** Which function writes the WKT2 of a raster into its STAC item when there is no EPSG code? Which flag does `curate` raise for a raster whose CRS differs from the site's, and in which function? Use `inspect` or search the files.

In [ ]:
import inspect
from data_acquisition.catalog import stac
from data_acquisition.steps import curate
...  # your code here